# Pedir datos a la API por rango de tiempo

Prueba cómo se piden lecturas entre dos horas con `fetch(start, end)` de `api_client.py`.

- Con `USAR_API_REAL = False` lee los datos simulados (`SimulatedSource`), que se comportan igual que la API.
- Con `USAR_API_REAL = True` llama a la API real (`ApiSource`). Para eso hace falta la IP de la Raspberry Pi en `config.yaml` y estar en la misma red.

Reglas del contrato (`platform/docs/api/openapi.yaml`): horas en Europe/Amsterdam, los dos extremos incluidos, como máximo 24 h por petición y 404 si ningún sensor tiene datos en ese rango.

Al final (paso 7) se escuchan los mensajes MQTT que envían los nodos, para comparar lo que llega a la Raspberry Pi con lo que devuelve la API.

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import yaml

# Buscar la carpeta del repositorio (la que tiene config.yaml) y añadir src
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("No se encuentra config.yaml: abre el notebook dentro del repositorio")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from occupancy.api_client import ApiSource, SimulatedSource, format_time

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

## Paso 1: elegir de dónde vienen los datos

Las dos fuentes tienen el mismo método `fetch(start, end)` y devuelven lo mismo: el campo `data` de la respuesta, o `None` si no hay datos.

In [2]:
USAR_API_REAL = True 

if USAR_API_REAL:
    source = ApiSource(config)
    print("API real:", source.url)
else:
    source = SimulatedSource(ROOT / "data" / "simulated" / "readings.json")
    inicio_datos, fin_datos = source.time_span()
    print("Datos simulados desde", format_time(inicio_datos), "hasta", format_time(fin_datos))

API real: http://192.168.0.181:8080/api/v1/readings/designlab/2


## Paso 2: elegir el rango

Escribe el inicio y el fin como texto, `"AAAA-MM-DD HH:MM"` o `"AAAA-MM-DD HH:MM:SS"`, en hora de Ámsterdam. Hay un rango de ejemplo para cada fuente: con la API real, la primera sesión real (4 nov); con los datos simulados, dentro de lo que imprime el paso 1.

Si prefieres pedir los últimos minutos (útil con la API real), pon `ULTIMOS_MINUTOS` a un número y se ignoran `INICIO` y `FIN`:

- Con la API real, "ahora" es la hora de Ámsterdam (la que espera la API), no la del portátil. Como en `predict.py`, el fin se alarga `clock_margin_minutes` (de `config.yaml`) por si el reloj de la Raspberry Pi va adelantado.
- Con los datos simulados, "ahora" es el final de los datos.

In [3]:
from datetime import timedelta
from zoneinfo import ZoneInfo

AMSTERDAM = ZoneInfo("Europe/Amsterdam")

if USAR_API_REAL:
    INICIO = "2026-11-04 15:20"   # primera sesión real (designlab / sala 2)
    FIN = "2026-11-04 15:35"
else:
    INICIO = "2026-10-20 10:00"   # dentro de los datos simulados
    FIN = "2026-10-20 10:15"
ULTIMOS_MINUTOS = 15   # por ejemplo 15 para pedir los últimos 15 minutos; None para usar INICIO y FIN


def leer_hora(texto):
    """Convierte 'AAAA-MM-DD HH:MM' o 'AAAA-MM-DD HH:MM:SS' en datetime."""
    for formato in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try:
            return datetime.strptime(texto.strip(), formato)
        except ValueError:
            pass
    raise ValueError(f"Hora no válida: {texto!r}. Usa 'AAAA-MM-DD HH:MM' o 'AAAA-MM-DD HH:MM:SS'")


if ULTIMOS_MINUTOS is not None:
    if USAR_API_REAL:
        # Hora de Ámsterdam sin zona horaria, como la espera la API
        ahora = datetime.now(AMSTERDAM).replace(tzinfo=None)
        margen = timedelta(minutes=config["prediction"]["clock_margin_minutes"])
    else:
        ahora = fin_datos
        margen = timedelta(0)
    start = ahora - timedelta(minutes=ULTIMOS_MINUTOS)
    end = ahora + margen
else:
    start = leer_hora(INICIO)
    end = leer_hora(FIN)

# Comprobaciones antes de llamar
if end <= start:
    raise ValueError("FIN tiene que ser después de INICIO")
if end - start > timedelta(hours=24):
    print("AVISO: más de 24 h; la API real responderá con error 400")
if not USAR_API_REAL and (end < inicio_datos or start > fin_datos):
    print("AVISO: el rango está fuera de los datos simulados; fetch devolverá None")

print("start_time =", format_time(start))
print("end_time   =", format_time(end))
print("duración   =", end - start)

start_time = 2026-11-04 16:21:03.129
end_time   = 2026-11-04 16:46:03.129
duración   = 0:25:00


## Paso 3: pedir los datos

Se muestran igual que en `predict.py` (mismas funciones `show_request` y `show_response`): la petición y, por cada sensor (columnas), cuántas lecturas llegaron, la hora de la primera y la última y el último valor. Para los PIR, el último valor es la actividad de cada uno separada por `/`.

In [4]:
from occupancy.predict import show_request, show_response

data = source.fetch(start, end)
show_request(source)
show_response(source, data)

  1. REQUEST TO THE API
     endpoint: GET http://192.168.0.181:8080/api/v1/readings/designlab/2
     from:     2026-11-04 16:21:03.129
     to:       2026-11-04 16:46:03.129
     sensors:  pirs, pressure, humidity, temperature, co2Equivalent, voc, iaq, mic_vol, wifiPackets, bleAdverts, busy_level
  2. API RESPONSE: HTTP 200
                     pirs  pressure  humidity temperature co2Equivalent       voc       iaq   mic_vol wifiPackets bleAdverts busy_level
     readings          89        89        89          89            89        89        89        89          89         89         89
     first       16:21:07  16:21:07  16:21:07    16:21:07      16:21:07  16:21:07  16:21:07  16:21:07    16:21:07   16:21:07   16:21:07
     last        16:35:49  16:35:49  16:35:49    16:35:49      16:35:49  16:35:49  16:35:49  16:35:49    16:35:49   16:35:49   16:35:49
     last value   50/0/50   1027.88     34.76       24.58         742.5      0.87    105.51    -31.87         445        193     

Con una lectura cada 10 s, 15 minutos dan unas 90 lecturas (91 si el intervalo fuera exacto, porque los dos extremos se incluyen). En los datos reales el intervalo no es exacto (entre 6 y 29 s en la primera sesión), así que el número cambia un poco; si se usa el margen del reloj, el fin cae en el futuro y solo llegan las lecturas hasta ahora.

## Paso 4: ver un sensor normal como tabla

Cada lectura es un diccionario con `timestamp` y `value`.

In [5]:
sensor = "temperature"
if data is None:
    print("No hay datos: cambia el rango en el paso 2")
else:
    display(pd.DataFrame(data[sensor]).head())

,timestamp,value
0,2026-11-04 16:21:07.251,24.56
1,2026-11-04 16:21:17.272,24.57
2,2026-11-04 16:21:27.299,24.57
3,2026-11-04 16:21:37.346,24.57
4,2026-11-04 16:21:47.409,24.57


## Paso 5: ver los PIR

Su estructura es distinta: cada lectura trae `values`, una lista con el `ID` y la `activity` (0 a 100) de cada PIR.

In [6]:
if data is None:
    print("No hay datos: cambia el rango en el paso 2")
else:
    print(data["pirs"][:2])

[{'timestamp': '2026-11-04 16:21:07.251', 'values': [{'ID': 1, 'activity': 20}, {'ID': 2, 'activity': 0}, {'ID': 3, 'activity': 60}]}, {'timestamp': '2026-11-04 16:21:17.272', 'values': [{'ID': 1, 'activity': 20}, {'ID': 2, 'activity': 0}, {'ID': 3, 'activity': 40}]}]


## Paso 6: comprobar los casos límite

- Un rango sin datos devuelve `None`.
- Con la API real, un rango de más de 24 h devuelve error 400 y se ve el mensaje de la API.

In [7]:
# Rango sin datos (otro día)
print(source.fetch(datetime(2026, 10, 21, 10, 0), datetime(2026, 10, 21, 10, 5)))

None


In [8]:
# Solo con la API real: más de 24 h
if USAR_API_REAL:
    try:
        source.fetch(datetime(2026, 11, 3, 10, 0), datetime(2026, 11, 4, 12, 0))
    except RuntimeError as error:
        print(error)
else:
    print("Esta prueba solo tiene sentido con la API real")

The API answered 400: Time range must be between 0 and 24 hours.


## Paso 7: ver los mensajes MQTT que envían los nodos

Esto no es la API: escucha directamente el broker MQTT de la Raspberry Pi (puerto 1883) durante 30 s, en el topic de la sala de `config.yaml` (`sensors/<edificio>/<sala>`). Sirve para comparar lo que envían los nodos con lo que devuelve la API.

Diferencia importante: en MQTT el `timestamp` va en milisegundos desde 1970 (UTC), mientras que la API lo devuelve como texto en hora de Ámsterdam. La celda muestra los dos.

Necesita `paho-mqtt` (está en `requirements.txt`).

In [9]:
import json
import time
from urllib.parse import urlparse

import paho.mqtt.client as mqtt

BROKER = urlparse(config["api"]["base_url"]).hostname   # la misma Raspberry Pi que la API
PUERTO = 1883
TOPIC = f"sensors/{config['api']['building']}/{config['api']['room']}"
SEGUNDOS = 30

mensajes = []

def al_recibir(client, userdata, msg):
    mensajes.append((msg.topic, msg.payload.decode(errors="replace")))

cliente = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
cliente.on_message = al_recibir
cliente.connect(BROKER, PUERTO)
cliente.subscribe(TOPIC)
cliente.loop_start()
time.sleep(SEGUNDOS)
cliente.loop_stop()
cliente.disconnect()

print(f"Mensajes recibidos en {SEGUNDOS} s en {TOPIC}:", len(mensajes))
for topic, payload in mensajes[:5]:
    print("\nTOPIC:", topic)
    print("PAYLOAD:", payload[:600])
    try:
        ms = json.loads(payload)["timestamp"]
        hora = datetime.fromtimestamp(ms / 1000, AMSTERDAM).replace(tzinfo=None)
        print("timestamp", ms, "->", format_time(hora), "(hora de Ámsterdam, como en la API)")
    except (ValueError, KeyError, TypeError):
        pass

Mensajes recibidos en 30 s en sensors/designlab/2: 3

TOPIC: sensors/designlab/2
PAYLOAD: {"temperature":24.59,"humidity":34.84,"pressure":1027.88,"iaq":109.44,"voc":0.9,"co2Equivalent":759.63,"pirs":[{"ID":1,"activity":40},{"ID":2,"activity":0},{"ID":3,"activity":30}],"mic_vol":-31.48,"wifiPackets":415,"bleAdverts":165,"busy_level":3,"timestamp":1793806569914}
timestamp 1793806569914 -> 2026-11-04 16:36:09.914 (hora de Ámsterdam, como en la API)

TOPIC: sensors/designlab/2
PAYLOAD: {"temperature":24.57,"humidity":34.87,"pressure":1027.88,"iaq":108.61,"voc":0.89,"co2Equivalent":756.01,"pirs":[{"ID":1,"activity":30},{"ID":2,"activity":0},{"ID":3,"activity":40}],"mic_vol":-34.63,"wifiPackets":366,"bleAdverts":160,"busy_level":3,"timestamp":1793806579950}
timestamp 1793806579950 -> 2026-11-04 16:36:19.950 (hora de Ámsterdam, como en la API)

TOPIC: sensors/designlab/2
PAYLOAD: {"temperature":24.56,"humidity":34.87,"pressure":1027.88,"iaq":107.49,"voc":0.88,"co2Equivalent":751.12,"pirs":[{